## 1 - Packages

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, make_scorer
from sklearn.svm import SVC
from imblearn.over_sampling import RandomOverSampler
from imblearn.pipeline import Pipeline as ImbPipeline
import joblib
import warnings
warnings.filterwarnings('ignore')
from time import time

%matplotlib inline

print("Packages imported successfully!")

Packages imported successfully!


## 2 - Load Preprocessed Data

In [2]:
# Load preprocessed data
X = joblib.load('../data/processed/X_vectorized.pkl')
Y = joblib.load('../data/processed/Y_labels.pkl')
vectorizer = joblib.load('../data/processed/vectorizer.pkl')
label_encoder = joblib.load('../data/processed/label_encoder.pkl')

print(f"X shape: {X.shape}")
print(f"Y shape: {Y.shape}")
print(f"Number of features: {X.shape[1]}")
print(f"Number of samples: {X.shape[0]}")
print(f"\nSparse matrix format: {type(X)}")

X shape: (8462, 90493)
Y shape: (8462,)
Number of features: 90493
Number of samples: 8462

Sparse matrix format: <class 'scipy.sparse._csr.csr_matrix'>


## 3 - Prepare Binary Dimensions

In [3]:
# Extract binary labels for each dimension
personality_types = label_encoder.inverse_transform(Y)

dimensions = {
    'I/E': np.array([1 if pt[0] == 'I' else 0 for pt in personality_types]),
    'N/S': np.array([1 if pt[1] == 'N' else 0 for pt in personality_types]),
    'T/F': np.array([1 if pt[2] == 'T' else 0 for pt in personality_types]),
    'J/P': np.array([1 if pt[3] == 'J' else 0 for pt in personality_types])
}

# Print distributions
print("=" * 60)
print("DIMENSION DISTRIBUTIONS")
print("=" * 60)
for dim_name, labels in dimensions.items():
    pos_class = dim_name.split('/')[0]
    neg_class = dim_name.split('/')[1]
    n_pos = np.sum(labels == 1)
    n_neg = np.sum(labels == 0)
    pct_pos = 100 * n_pos / len(labels)
    print(f"{dim_name}: {pos_class}={n_pos} ({pct_pos:.1f}%), {neg_class}={n_neg} ({100-pct_pos:.1f}%)")
print("=" * 60)

DIMENSION DISTRIBUTIONS
I/E: I=6534 (77.2%), E=1928 (22.8%)
N/S: N=7284 (86.1%), S=1178 (13.9%)
T/F: T=3907 (46.2%), F=4555 (53.8%)
J/P: J=3365 (39.8%), P=5097 (60.2%)


## 4 - GridSearchCV Configuration

**Strategy: Hyperparameter Tuning with Upsampling Only**

We skip non-upsampled tuning because:
- **Problem**: Non-upsampled models achieve high F1 (~86%) but LOW specificity (~7%) = lazy model
- **Root cause**: Model learns "always predict majority class" which inflates metrics
- **Solution**: Upsampling forces model to learn BOTH classes, improving specificity to ~40-50%
- **Key insight**: Optimal C values don't change significantly with upsampling, so we tune directly on upsampled data

**Parameter Grid (Optimized for Speed):**
- **C**: [0.001, 0.01, 0.1, 0.5] - Reduced to 4 values (kept lower range)
- **kernel**: ['linear'] only - Linear kernel sufficient for high-dimensional text data

- RBF and Poly kernels removed for computational efficiency**Total**: 4 C values × 4 dimensions = 16 combinations (fast!)



**Cross-Validation:**- **Pipeline approach**: Upsampling happens inside each CV fold to prevent data leakage

- 5-fold StratifiedKFold (preserves class proportions)- Scoring: F1-score (better for imbalanced data than accuracy)

In [4]:
# Configuration
RANDOM_STATE = 42
MAX_ITER = 1000
N_JOBS = -1  # Use all CPU cores

# Define parameter grids for different kernel types
# DRASTICALLY REDUCED FOR SPEED
param_grids = {
    'linear': {
        'C': [0.001, 0.01, 0.1, 0.5],  # Reduced from 9 to 5 (kept lower range)
        'kernel': ['linear']
    }
    # ,
    # 'rbf': {
    #     'C': [0.01, 0.1, 1.0],  # Reduced from 7 to 3
    #     'kernel': ['rbf'],
    #     'gamma': ['scale', 0.01, 0.1]  # Reduced from 6 to 3
    # }
    # # Poly kernel REMOVED entirely for speed
}

# Cross-validation strategy
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

print("Configuration complete! (REDUCED GRID FOR SPEED)")
print(f"\nLinear kernel: {len(param_grids['linear']['C'])} C values to test")
# print(f"RBF kernel: {len(param_grids['rbf']['C'])} C × {len(param_grids['rbf']['gamma'])} gamma = {len(param_grids['rbf']['C']) * len(param_grids['rbf']['gamma'])} combinations")
print(f"\nTotal combinations per dimension:")
# print(f"  Without upsampling: {len(param_grids['linear']['C']) + len(param_grids['rbf']['C']) * len(param_grids['rbf']['gamma'])} = 14 (was 75)")
print(f"  With upsampling: Same 14")
print(f"\n⚡ Speed improvement: ~5x faster!")
print(f"\n💡 TIP: If still too slow, comment out 'rbf' from param_grids dict to test linear only")

Configuration complete! (REDUCED GRID FOR SPEED)

Linear kernel: 4 C values to test

Total combinations per dimension:
  With upsampling: Same 14

⚡ Speed improvement: ~5x faster!

💡 TIP: If still too slow, comment out 'rbf' from param_grids dict to test linear only


## 5 - Hyperparameter Tuning (With Upsampling)

Using Pipeline with RandomOverSampler to handle class imbalance properly.

- Better real-world performance across all personality types

**Key Benefits:**- Improves specificity (minority class detection) from ~7% to ~40-50%

- Upsampling happens **inside each CV fold** (prevents data leakage)- Model learns to recognize BOTH majority and minority classes

In [5]:
# Store all results
all_results = []

In [6]:
print("=" * 80)
print("STARTING HYPERPARAMETER TUNING (With Upsampling)")
print("=" * 80)
print("Note: Using Pipeline to prevent data leakage (upsampling happens inside each CV fold)")
print("=" * 80)

for dim_name, Y_binary in dimensions.items():
    print(f"\n{'=' * 80}")
    print(f"DIMENSION: {dim_name}")
    print("=" * 80)
    
    # Split data once (80/20 train/test)
    X_train, X_test, y_train, y_test = train_test_split(
        X, Y_binary, test_size=0.2, random_state=RANDOM_STATE, stratify=Y_binary
    )
    
    print(f"Train size: {X_train.shape[0]}, Test size: {X_test.shape[0]}")
    
    # Test each kernel type
    for kernel_name, param_grid in param_grids.items():
        print(f"\n  Testing {kernel_name.upper()} kernel with upsampling...")
        start_time = time()
        
        # Create pipeline with upsampling + SVM
        pipeline = ImbPipeline([
            ('sampler', RandomOverSampler(random_state=RANDOM_STATE)),
            ('classifier', SVC(max_iter=MAX_ITER, random_state=RANDOM_STATE))
        ])
        
        # Adjust parameter grid for pipeline (need 'classifier__' prefix)
        pipeline_param_grid = {f'classifier__{key}': value for key, value in param_grid.items()}
        
        # GridSearchCV
        grid_search = GridSearchCV(
            estimator=pipeline,
            param_grid=pipeline_param_grid,
            cv=cv,
            scoring='f1',
            n_jobs=N_JOBS,
            verbose=0,
            return_train_score=True
        )
        
        grid_search.fit(X_train, y_train)
        elapsed = time() - start_time
        
        # Get best model
        best_pipeline = grid_search.best_estimator_
        best_params = grid_search.best_params_
        best_cv_score = grid_search.best_score_
        
        # Extract actual params (remove 'classifier__' prefix)
        actual_params = {key.replace('classifier__', ''): value for key, value in best_params.items()}
        
        # Evaluate on test set
        y_pred = best_pipeline.predict(X_test)
        test_accuracy = accuracy_score(y_test, y_pred)
        test_f1 = f1_score(y_test, y_pred, zero_division=0)
        test_precision = precision_score(y_test, y_pred, zero_division=0)
        test_recall = recall_score(y_test, y_pred, zero_division=0)
        
        # Get classifier from pipeline
        best_classifier = best_pipeline.named_steps['classifier']
        
        # Store results
        result = {
            'dimension': dim_name,
            'kernel': kernel_name,
            'upsampling': True,
            'best_C': actual_params['C'],
            'best_gamma': actual_params.get('gamma', None),
            'best_degree': actual_params.get('degree', None),
            'cv_f1_score': best_cv_score,
            'test_accuracy': test_accuracy,
            'test_f1_score': test_f1,
            'test_precision': test_precision,
            'test_recall': test_recall,
            'fit_time_seconds': elapsed,
            'n_support_vectors': sum(best_classifier.n_support_) if hasattr(best_classifier, 'n_support_') else None
        }
        all_results.append(result)
        
        print(f"    Best params: {actual_params}")
        print(f"    CV F1: {best_cv_score:.4f}")
        print(f"    Test F1: {test_f1:.4f} | Test Acc: {test_accuracy:.4f}")
        print(f"    Time: {elapsed:.1f}s")

print(f"\n{'=' * 80}")
print("✅ HYPERPARAMETER TUNING COMPLETE (With Upsampling)")
print("=" * 80)

STARTING HYPERPARAMETER TUNING (With Upsampling)
Note: Using Pipeline to prevent data leakage (upsampling happens inside each CV fold)

DIMENSION: I/E
Train size: 6769, Test size: 1693

  Testing LINEAR kernel with upsampling...
    Best params: {'C': 0.5, 'kernel': 'linear'}
    CV F1: 0.8548
    Test F1: 0.8547 | Test Acc: 0.7507
    Time: 199.0s

DIMENSION: N/S
Train size: 6769, Test size: 1693

  Testing LINEAR kernel with upsampling...
    Best params: {'C': 0.5, 'kernel': 'linear'}
    CV F1: 0.9074
    Test F1: 0.9106 | Test Acc: 0.8376
    Time: 214.4s

DIMENSION: T/F
Train size: 6769, Test size: 1693

  Testing LINEAR kernel with upsampling...
    Best params: {'C': 0.5, 'kernel': 'linear'}
    CV F1: 0.6356
    Test F1: 0.6290 | Test Acc: 0.4808
    Time: 164.8s

DIMENSION: J/P
Train size: 6769, Test size: 1693

  Testing LINEAR kernel with upsampling...
    Best params: {'C': 0.5, 'kernel': 'linear'}
    CV F1: 0.5627
    Test F1: 0.5676 | Test Acc: 0.4259
    Time: 174.6s



## 6 - Results Summary (DataFrame)

In [7]:
# Convert results to DataFrame
results_df = pd.DataFrame(all_results)

# Round numerical columns for readability
numerical_cols = ['cv_f1_score', 'test_accuracy', 'test_f1_score', 'test_precision', 'test_recall', 'fit_time_seconds']
results_df[numerical_cols] = results_df[numerical_cols].round(4)

# Sort by dimension and test F1 score
results_df = results_df.sort_values(['dimension', 'test_f1_score'], ascending=[True, False])

print("=" * 100)
print("COMPLETE RESULTS SUMMARY")
print("=" * 100)
display(results_df)

# Save to CSV
results_df.to_csv('svm_hyperparameter_tuning_results.csv', index=False)
print("\n✅ Results saved to 'svm_hyperparameter_tuning_results.csv'")

COMPLETE RESULTS SUMMARY


,dimension,kernel,upsampling,best_C,best_gamma,best_degree,cv_f1_score,test_accuracy,test_f1_score,test_precision,test_recall,fit_time_seconds,n_support_vectors
0,I/E,linear,True,0.5,None,None,0.8548,0.7507,0.8547,0.7771,0.9495,199.0287,2000
3,J/P,linear,True,0.5,None,None,0.5627,0.4259,0.5676,0.4051,0.9480,174.6004,2000
1,N/S,linear,True,0.5,None,None,0.9074,0.8376,0.9106,0.8648,0.9616,214.3522,2000
2,T/F,linear,True,0.5,None,None,0.6356,0.4808,0.6290,0.4694,0.9527,164.7732,2000



✅ Results saved to 'svm_hyperparameter_tuning_results.csv'


## 7 - Best Model per Dimension

In [8]:
# Find best model for each dimension (by test F1 score)
best_models = results_df.loc[results_df.groupby('dimension')['test_f1_score'].idxmax()]

print("=" * 100)
print("BEST MODEL PER DIMENSION (Highest Test F1-Score)")
print("=" * 100)
display(best_models[['dimension', 'kernel', 'upsampling', 'best_C', 'best_gamma', 'best_degree', 
                      'cv_f1_score', 'test_f1_score', 'test_accuracy', 'test_recall', 'fit_time_seconds']])

print("\n" + "=" * 100)
print("KEY INSIGHTS:")
print("=" * 100)
for idx, row in best_models.iterrows():
    print(f"\n{row['dimension']}:")
    print(f"  Best kernel: {row['kernel'].upper()}")
    print(f"  Upsampling: {'Yes' if row['upsampling'] else 'No'}")
    print(f"  C = {row['best_C']}")
    if row['best_gamma'] is not None:
        print(f"  gamma = {row['best_gamma']}")
    if row['best_degree'] is not None:
        print(f"  degree = {row['best_degree']}")
    print(f"  Test F1: {row['test_f1_score']:.4f}")
    print(f"  Test Accuracy: {row['test_accuracy']:.4f}")

BEST MODEL PER DIMENSION (Highest Test F1-Score)


,dimension,kernel,upsampling,best_C,best_gamma,best_degree,cv_f1_score,test_f1_score,test_accuracy,test_recall,fit_time_seconds
0,I/E,linear,True,0.5,None,None,0.8548,0.8547,0.7507,0.9495,199.0287
3,J/P,linear,True,0.5,None,None,0.5627,0.5676,0.4259,0.9480,174.6004
1,N/S,linear,True,0.5,None,None,0.9074,0.9106,0.8376,0.9616,214.3522
2,T/F,linear,True,0.5,None,None,0.6356,0.6290,0.4808,0.9527,164.7732



KEY INSIGHTS:

I/E:
  Best kernel: LINEAR
  Upsampling: Yes
  C = 0.5
  Test F1: 0.8547
  Test Accuracy: 0.7507

J/P:
  Best kernel: LINEAR
  Upsampling: Yes
  C = 0.5
  Test F1: 0.5676
  Test Accuracy: 0.4259

N/S:
  Best kernel: LINEAR
  Upsampling: Yes
  C = 0.5
  Test F1: 0.9106
  Test Accuracy: 0.8376

T/F:
  Best kernel: LINEAR
  Upsampling: Yes
  C = 0.5
  Test F1: 0.6290
  Test Accuracy: 0.4808


## 8 - Export Optimal Parameters (JSON)

**This is the most important output!** Save optimal hyperparameters to JSON file for use in main SVM analysis notebook.

In [9]:
# Create dictionary of optimal hyperparameters for each dimension
optimal_params = {}

for idx, row in best_models.iterrows():
    optimal_params[row['dimension']] = {
        'best_C': float(row['best_C']),  # Convert to native Python float
        'kernel': row['kernel'],
        'gamma': row['best_gamma'] if pd.notna(row['best_gamma']) else None,
        'degree': int(row['best_degree']) if pd.notna(row['best_degree']) else None,
        'max_iter': 1000,
        'random_state': 42,
        'upsampling': bool(row['upsampling']),
        # Performance metrics for reference
        'expected_test_f1': float(row['test_f1_score']),
        'expected_test_accuracy': float(row['test_accuracy'])
    }

# Save to JSON
import json
import os

# Create models directory if it doesn't exist
os.makedirs('../models', exist_ok=True)

with open('../models/optimal_hyperparameters_svm.json', 'w') as f:
    json.dump(optimal_params, f, indent=2)

print("=" * 100)
print("✅ OPTIMAL HYPERPARAMETERS EXPORTED")
print("=" * 100)
print("\nSaved to: ../models/optimal_hyperparameters_svm.json")
print("\nContents:")
print(json.dumps(optimal_params, indent=2))
print("\n" + "=" * 100)
print("📝 USE THESE PARAMETERS IN YOUR MAIN mbti_svm.ipynb NOTEBOOK!")
print("=" * 100)

✅ OPTIMAL HYPERPARAMETERS EXPORTED

Saved to: ../models/optimal_hyperparameters_svm.json

Contents:
{
  "I/E": {
    "best_C": 0.5,
    "kernel": "linear",
    "gamma": null,
    "degree": null,
    "max_iter": 1000,
    "random_state": 42,
    "upsampling": true,
    "expected_test_f1": 0.8547,
    "expected_test_accuracy": 0.7507
  },
  "J/P": {
    "best_C": 0.5,
    "kernel": "linear",
    "gamma": null,
    "degree": null,
    "max_iter": 1000,
    "random_state": 42,
    "upsampling": true,
    "expected_test_f1": 0.5676,
    "expected_test_accuracy": 0.4259
  },
  "N/S": {
    "best_C": 0.5,
    "kernel": "linear",
    "gamma": null,
    "degree": null,
    "max_iter": 1000,
    "random_state": 42,
    "upsampling": true,
    "expected_test_f1": 0.9106,
    "expected_test_accuracy": 0.8376
  },
  "T/F": {
    "best_C": 0.5,
    "kernel": "linear",
    "gamma": null,
    "degree": null,
    "max_iter": 1000,
    "random_state": 42,
    "upsampling": true,
    "expected_test_f1": 